## Load libraries

In [1]:
import numpy as np
import pandas as pd
import time
import random

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid

from statsmodels.tsa.statespace.sarimax import SARIMAX

from joblib import Parallel, delayed
import threading

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# silence only SARIMAX convergence warnings
warnings.simplefilter("ignore", ConvergenceWarning)
warnings.filterwarnings("ignore", message="Maximum Likelihood optimization failed to converge")


## Config

In [2]:
# ---------------- CONFIG ----------------
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# tuning window: same as other models
TUNE_START_DATE = pd.Timestamp("2007-04-01")
TUNE_END_DATE   = pd.Timestamp("2022-03-31")

# rolling CV: 10 years train, 1 year val (in months)
ROLLING_TRAIN_WINDOW = 120   # 120 months
ROLLING_VAL_WINDOW   = 12    # 12 months

# SARIMAX hyperparameter grid
# (keep this small-ish or it will be slow)
param_grid = list(ParameterGrid({
    "order_p":  [0, 1, 2],
    "order_d":  [0, 1],        # usually 1 for house prices
    "order_q":  [0, 1, 2],
    "seasonal_P": [0, 1],
    "seasonal_D": [0, 1],      # seasonal differencing (12)
    "seasonal_Q": [0, 1],
    "seasonal_period": [12]
}))

# --- Force (2,0,2)(0,0,0,12) to run first ---
preferred = {
    "order_p": 2,
    "order_d": 0,
    "order_q": 2,
    "seasonal_P": 0,
    "seasonal_D": 0,
    "seasonal_Q": 0,
    "seasonal_period": 12,
}

# move preferred config to front if present
if preferred in param_grid:
    param_grid = [preferred] + [p for p in param_grid if p != preferred]
else:
    # if it's missing for some reason, insert it at the front
    param_grid = [preferred] + param_grid

print("First SARIMAX config to run:", param_grid[0])

print(f"Number of SARIMAX configs: {len(param_grid)}")

# feature lists (same as before)
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
]

base_feature_cols = continuous_cols + categorical_cols

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

First SARIMAX config to run: {'order_p': 2, 'order_d': 0, 'order_q': 2, 'seasonal_P': 0, 'seasonal_D': 0, 'seasonal_Q': 0, 'seasonal_period': 12}
Number of SARIMAX configs: 144


## Metric functions

In [3]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(
        100.0 * np.mean(
            2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
        )
    )

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """
    MASE using seasonal naive (lag m) error on TRAIN period.
    y_train should be the *original* target in the same units.
    """
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)

    if len(y_train) <= m:
        return np.nan

    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)

## Load data

In [4]:
df = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

mask_tune = (df[TIME_COL] >= TUNE_START_DATE) & (df[TIME_COL] <= TUNE_END_DATE)
df = df.loc[mask_tune].copy()

# ensure one row per (Date, AreaCode)
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)

print(f"Tuning period: {dates[0].date()} → {dates[-1].date()}")
print("Total months:", T_total)
print("Number of LAs:", N)

Tuning period: 2007-04-01 → 2022-03-01
Total months: 180
Number of LAs: 294


In [5]:
full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# ffill/bfill features + target within each LA
df_panel[base_feature_cols + [TARGET_COL]] = (
    df_panel[base_feature_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

# last-resort fill
missing_total = df_panel[base_feature_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after ffill/bfill. Filling with column means.")
    col_means = df_panel[base_feature_cols + [TARGET_COL]].mean()
    df_panel[base_feature_cols + [TARGET_COL]] = df_panel[base_feature_cols + [TARGET_COL]].fillna(col_means)

print("NaNs after panel completion:",
      df_panel[base_feature_cols + [TARGET_COL]].isna().sum().sum())

NaNs after panel completion: 0


## Rolling origin and folds

In [6]:
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW

while True:
    train_end_idx = start_idx      # exclusive
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > T_total:
        break

    train_start_date = dates[train_end_idx - ROLLING_TRAIN_WINDOW]
    train_end_date   = dates[train_end_idx - 1]
    val_start_date   = dates[val_start_idx]
    val_end_date     = dates[val_end_idx - 1]

    fold_specs.append((train_end_idx, val_start_idx, val_end_idx,
                       train_start_date, train_end_date,
                       val_start_date, val_end_date))

    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")
for i, (_, _, _, ts, te, vs, ve) in enumerate(fold_specs, start=1):
    print(f"  Fold {i}: Train {ts:%Y-%m}–{te:%Y-%m}, Val {vs:%Y-%m}–{ve:%Y-%m}")


Number of folds: 5
  Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03
  Fold 2: Train 2008-04–2018-03, Val 2018-04–2019-03
  Fold 3: Train 2009-04–2019-03, Val 2019-04–2020-03
  Fold 4: Train 2010-04–2020-03, Val 2020-04–2021-03
  Fold 5: Train 2011-04–2021-03, Val 2021-04–2022-03


## Tuning

In [7]:
## ========= TUNING: SARIMAX PANEL (ALL LAs) ========= ##

# 1) Precompute scaled train/val data ONCE per fold (uses ALL LAs)
fold_data = []  # one dict per fold

for fold_no, (train_end_idx, val_start_idx, val_end_idx,
              train_start_date, train_end_date,
              val_start_date, val_end_date) in enumerate(fold_specs, start=1):

    print(f"Precomputing fold {fold_no}: "
          f"Train {train_start_date:%Y-%m}–{train_end_date:%Y-%m}, "
          f"Val {val_start_date:%Y-%m}–{val_end_date:%Y-%m}")

    train_dates = dates[:train_end_idx]
    val_dates   = dates[val_start_idx:val_end_idx]

    df_train = df_panel.loc[(train_dates, slice(None)), :].copy()
    df_val   = df_panel.loc[(val_dates,   slice(None)), :].copy()

    # Exogenous scaler (continuous features)
    exog_scaler = StandardScaler()
    exog_scaler.fit(df_train[continuous_cols])

    df_train_scaled = df_train.copy()
    df_val_scaled   = df_val.copy()
    df_train_scaled.loc[:, continuous_cols] = exog_scaler.transform(df_train[continuous_cols])
    df_val_scaled.loc[:,   continuous_cols] = exog_scaler.transform(df_val[continuous_cols])

    # Target scaler (train only – no leakage)
    target_scaler = StandardScaler()
    target_scaler.fit(df_train[[TARGET_COL]])
    df_train_scaled[TARGET_COL] = target_scaler.transform(df_train[[TARGET_COL]])
    df_val_scaled[TARGET_COL]   = target_scaler.transform(df_val[[TARGET_COL]])

    # Original train target for MASE baseline (all LAs, original units)
    y_train_orig = df_train[TARGET_COL].values

    fold_data.append({
        "df_train": df_train,
        "df_val": df_val,
        "df_train_scaled": df_train_scaled,
        "df_val_scaled": df_val_scaled,
        "target_scaler": target_scaler,
        "y_train_orig": y_train_orig,
    })

print(f"Precomputed data for {len(fold_data)} folds.")

Precomputing fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03
Precomputing fold 2: Train 2008-04–2018-03, Val 2018-04–2019-03
Precomputing fold 3: Train 2009-04–2019-03, Val 2019-04–2020-03
Precomputing fold 4: Train 2010-04–2020-03, Val 2020-04–2021-03
Precomputing fold 5: Train 2011-04–2021-03, Val 2021-04–2022-03
Precomputed data for 5 folds.


C:\Users\slong\AppData\Local\Temp\ipykernel_1608\3812654171.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.07376585 -1.07376585 -1.07376585 ...  2.30485596  2.30485596
  5.68347777]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_train_scaled.loc[:, continuous_cols] = exog_scaler.transform(df_train[continuous_cols])
C:\Users\slong\AppData\Local\Temp\ipykernel_1608\3812654171.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.07376585 -1.07376585 -1.07376585 ...  2.30485596  2.30485596
  5.68347777]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_val_scaled.loc[:,   continuous_cols] = exog_scaler.transform(df_val[continuous_cols])
C:\Users\slong\AppData\Local\Temp\ipykernel_1608\3812654171.py:26: FutureWarning: Setting an item of incompatibl

In [8]:
# Global best RMSE for early stopping across configs
best_rmse_mean = np.inf
best_rmse_lock = threading.Lock()

# NEW: global best first-fold AIC for early stopping
best_firstfold_aic = np.inf
best_firstfold_aic_lock = threading.Lock()

def evaluate_config(params):
    """
    Run rolling CV for a single SARIMAX config using
    precomputed fold_data and ALL LAs (no subsampling).
    Returns a dict with metrics, or None if no valid folds.
    """
    global best_rmse_mean

    print(f"\n=== Config {params} ===", flush=True)

    fold_mae_list   = []
    fold_rmse_list  = []
    fold_smape_list = []
    fold_mase_list  = []
    fold_aic_list   = []
    fold_bic_list   = []
    folds_used      = 0

    p  = params["order_p"]
    d  = params["order_d"]
    q  = params["order_q"]
    P  = params["seasonal_P"]
    D  = params["seasonal_D"]
    Q  = params["seasonal_Q"]
    s  = params["seasonal_period"]

    exog_cols = continuous_cols + categorical_cols

    for (fold_idx,
         (fold_spec, fd)) in enumerate(zip(fold_specs, fold_data), start=1):

        (train_end_idx, val_start_idx, val_end_idx,
         train_start_date, train_end_date,
         val_start_date, val_end_date) = fold_spec

        df_train        = fd["df_train"]
        df_val          = fd["df_val"]
        df_train_scaled = fd["df_train_scaled"]
        df_val_scaled   = fd["df_val_scaled"]
        target_scaler   = fd["target_scaler"]
        y_train_fold    = fd["y_train_orig"]  # for MASE (original units)

        print(
            f"  Fold {fold_idx}: Train {train_start_date:%Y-%m}–{train_end_date:%Y-%m}, "
            f"Val {val_start_date:%Y-%m}–{val_end_date:%Y-%m}",
            flush=True,
        )

        # Accumulators across LAs for this fold (original units)
        y_true_fold = []
        y_pred_fold = []
        aic_las     = []
        bic_las     = []

        for la in la_order:  # ALL LAs
            try:
                # scaled slices for modelling
                sub_train_scaled = df_train_scaled.xs(la, level=ENTITY_COL).sort_index()
                sub_val_scaled   = df_val_scaled.xs(la, level=ENTITY_COL).sort_index()
                # original val slice for metrics
                sub_val_orig     = df_val.xs(la, level=ENTITY_COL).sort_index()
            except KeyError:
                # LA missing entirely in this fold
                continue

            if sub_train_scaled[TARGET_COL].isna().all() or sub_val_scaled[TARGET_COL].isna().all():
                continue

            endog_train = sub_train_scaled[TARGET_COL].values.astype(float)   # scaled target
            exog_train  = sub_train_scaled[exog_cols].values.astype(float)
            exog_val    = sub_val_scaled[exog_cols].values.astype(float)
            n_val       = len(sub_val_scaled)
            y_true_orig = sub_val_orig[TARGET_COL].values.astype(float)      # original units

            # Safety check on series length
            if len(endog_train) < (p + P * s + 5):
                continue

            try:
                model = SARIMAX(
                    endog=endog_train,
                    exog=exog_train,
                    order=(p, d, q),
                    seasonal_order=(P, D, Q, s),
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                    simple_differencing=True,
                )

                res = model.fit(disp=False)

                # AIC/BIC for this LA in this fold
                aic_las.append(res.aic)
                bic_las.append(res.bic)

                # Forecast in scaled space
                y_forecast_scaled = res.forecast(steps=n_val, exog=exog_val)
                y_forecast_scaled = np.asarray(y_forecast_scaled, dtype=float).reshape(-1, 1)

                # Back to original £
                y_forecast = target_scaler.inverse_transform(y_forecast_scaled).flatten()

                y_true_fold.append(y_true_orig)
                y_pred_fold.append(y_forecast)

            except Exception:
                # convergence / numeric issues – skip this LA in this fold
                continue

        if not y_true_fold or not y_pred_fold:
            print("    ❌ No successful LA fits in this fold. Skipping fold.", flush=True)
            continue

        y_true_fold = np.concatenate(y_true_fold, axis=0)
        y_pred_fold = np.concatenate(y_pred_fold, axis=0)

        fold_mae   = mae(y_true_fold, y_pred_fold)
        fold_rmse  = rmse(y_true_fold, y_pred_fold)
        fold_smape = smape(y_true_fold, y_pred_fold)
        fold_mase  = mase(y_true_fold, y_pred_fold, y_train_fold, m=12)
        fold_aic   = float(np.mean(aic_las)) if aic_las else np.nan
        fold_bic   = float(np.mean(bic_las)) if bic_las else np.nan

        print(
            f"    Fold {fold_idx} MAE(£)={fold_mae:,.1f}, RMSE(£)={fold_rmse:,.1f}, "
            f"sMAPE={fold_smape:.3f}%, MASE={fold_mase:.3f}, "
            f"AIC={fold_aic:,.1f}, BIC={fold_bic:,.1f}",
            flush=True,
        )

        # ---------- NEW: AIC EARLY STOP AFTER FOLD 1 ----------
        if fold_idx == 1 and np.isfinite(fold_aic):
            global best_firstfold_aic
            with best_firstfold_aic_lock:
                if np.isfinite(best_firstfold_aic):
                    # If this config's first-fold AIC is much worse than the best so far, drop it.
                    # The margin (e.g. +50) is a tuning knob: smaller = stricter.
                    if fold_aic > best_firstfold_aic + 50.0:
                        print(
                            "    ⛔ Early stopping this config after fold 1 "
                            f"based on AIC (fold AIC={fold_aic:,.1f}, "
                            f"best first-fold AIC={best_firstfold_aic:,.1f}).",
                            flush=True,
                        )
                        return None  # abandon this config entirely
                    # Otherwise, if it's better, update the global best
                    elif fold_aic < best_firstfold_aic:
                        best_firstfold_aic = fold_aic
                else:
                    # first time we see a valid AIC
                    best_firstfold_aic = fold_aic

        fold_mae_list.append(fold_mae)
        fold_rmse_list.append(fold_rmse)
        fold_smape_list.append(fold_smape)
        fold_mase_list.append(fold_mase)
        fold_aic_list.append(fold_aic)
        fold_bic_list.append(fold_bic)
        folds_used += 1

        # -------- EARLY STOPPING AFTER AT LEAST 3 FOLDS --------
        if folds_used >= 3:
            partial_mean_rmse = float(np.mean(fold_rmse_list))

            with best_rmse_lock:
                current_best = best_rmse_mean

            if np.isfinite(current_best) and partial_mean_rmse > current_best * 1.2:
                print(
                    "    🔁 Early stopping this config after 3+ folds "
                    f"(partial RMSE {partial_mean_rmse:,.1f} worse than best {current_best:,.1f}).",
                    flush=True,
                )
                break
        # --------------------------------------------------------

    if folds_used == 0:
        print("  ❌ No valid folds for this config. Skipping.", flush=True)
        return None

    cfg_rmse_mean = float(np.mean(fold_rmse_list))

    cfg_result = {
        "model_type": "SARIMAX_panel",
        "order": (p, d, q),
        "seasonal_order": (P, D, Q, s),
        "folds_used": folds_used,

        # only the metrics you asked for (means over folds)
        "MAE_mean":   float(np.mean(fold_mae_list)),
        "RMSE_mean":  cfg_rmse_mean,
        "sMAPE_mean": float(np.mean(fold_smape_list)),
        "MASE_mean":  float(np.mean(fold_mase_list)),
        "AIC_mean":   float(np.nanmean(fold_aic_list)),
        "BIC_mean":   float(np.nanmean(fold_bic_list)),
    }

    # Update global best RMSE under lock (for early stopping)
    with best_rmse_lock:
        if cfg_rmse_mean < best_rmse_mean:
            print(f"  ✅ New best config with RMSE_mean={cfg_rmse_mean:,.1f}", flush=True)
            best_rmse_mean = cfg_rmse_mean

    return cfg_result


# 2) Run all configs in parallel
parallel_results = Parallel(
    n_jobs=8,          # adjust to taste; 8 matches your 7700X cores
    verbose=5,
    prefer="threads",  # threads so prints show nicely in Jupyter
)(
    delayed(evaluate_config)(params) for params in param_grid
)





=== Config {'order_p': 2, 'order_d': 0, 'order_q': 2, 'seasonal_P': 0, 'seasonal_D': 0, 'seasonal_Q': 0, 'seasonal_period': 12} ===

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 0, 'seasonal_D': 0, 'seasonal_P': 0, 'seasonal_Q': 0, 'seasonal_period': 12} ===

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 0, 'seasonal_D': 0, 'seasonal_P': 0, 'seasonal_Q': 1, 'seasonal_period': 12} ===

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 0, 'seasonal_D': 0, 'seasonal_P': 1, 'seasonal_Q': 0, 'seasonal_period': 12} ===

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 0, 'seasonal_D': 0, 'seasonal_P': 1, 'seasonal_Q': 1, 'seasonal_period': 12} ===

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 0, 'seasonal_D': 1, 'seasonal_P': 0, 'seasonal_Q': 0, 'seasonal_period': 12} ===
  Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 0, 'seasonal_D': 1, 'seasonal_P': 0, 'seasonal_Q': 1, 'seasonal_period': 12} ===
  Fold 1: Train 

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


    Fold 1 MAE(£)=108,733.2, RMSE(£)=168,742.8, sMAPE=38.751%, MASE=1.222, AIC=-409.0, BIC=-360.9
  Fold 2: Train 2008-04–2018-03, Val 2018-04–2019-03
    Fold 1 MAE(£)=6,175.0, RMSE(£)=12,679.6, sMAPE=2.257%, MASE=0.069, AIC=-525.8, BIC=-475.7
  Fold 2: Train 2008-04–2018-03, Val 2018-04–2019-03
    Fold 1 MAE(£)=6,083.3, RMSE(£)=12,791.2, sMAPE=2.195%, MASE=0.068, AIC=-574.7, BIC=-513.9
  Fold 2: Train 2008-04–2018-03, Val 2018-04–2019-03
    Fold 2 MAE(£)=109,667.4, RMSE(£)=168,117.8, sMAPE=38.891%, MASE=1.181, AIC=-462.4, BIC=-412.4
  Fold 3: Train 2009-04–2019-03, Val 2019-04–2020-03
    Fold 2 MAE(£)=6,644.8, RMSE(£)=16,163.1, sMAPE=2.286%, MASE=0.072, AIC=-586.9, BIC=-535.2
  Fold 3: Train 2009-04–2019-03, Val 2019-04–2020-03
    Fold 1 MAE(£)=108,136.5, RMSE(£)=168,299.4, sMAPE=38.514%, MASE=1.216, AIC=-395.0, BIC=-346.3
    ⛔ Early stopping this config after fold 1 based on AIC (fold AIC=-395.0, best first-fold AIC=-574.7).

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 0

[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed: 16.9min


    Fold 1 MAE(£)=108,136.5, RMSE(£)=167,789.4, sMAPE=38.531%, MASE=1.216, AIC=-394.5, BIC=-346.0
    ⛔ Early stopping this config after fold 1 based on AIC (fold AIC=-394.5, best first-fold AIC=-574.7).

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 1, 'seasonal_D': 0, 'seasonal_P': 0, 'seasonal_Q': 1, 'seasonal_period': 12} ===
  Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03
    Fold 1 MAE(£)=6,334.0, RMSE(£)=13,139.4, sMAPE=2.319%, MASE=0.071, AIC=-471.9, BIC=-421.1
    ⛔ Early stopping this config after fold 1 based on AIC (fold AIC=-471.9, best first-fold AIC=-574.7).

=== Config {'order_d': 0, 'order_p': 0, 'order_q': 1, 'seasonal_D': 0, 'seasonal_P': 1, 'seasonal_Q': 0, 'seasonal_period': 12} ===
  Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03
    Fold 1 MAE(£)=6,287.8, RMSE(£)=13,080.7, sMAPE=2.296%, MASE=0.071, AIC=-470.8, BIC=-417.3
    ⛔ Early stopping this config after fold 1 based on AIC (fold AIC=-470.8, best first-fold AIC=-574.7).

=== Config {'order_d': 

[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed: 168.7min


    Fold 1 MAE(£)=108,151.2, RMSE(£)=167,163.1, sMAPE=38.556%, MASE=1.216, AIC=-426.7, BIC=-373.1
    ⛔ Early stopping this config after fold 1 based on AIC (fold AIC=-426.7, best first-fold AIC=-576.4).

=== Config {'order_d': 0, 'order_p': 2, 'order_q': 1, 'seasonal_D': 1, 'seasonal_P': 1, 'seasonal_Q': 1, 'seasonal_period': 12} ===
  Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03
    Fold 1 MAE(£)=67,531,107,819,565,331,472,076,836,716,133,386,027,008.0, RMSE(£)=3,380,318,321,186,883,391,929,396,661,688,754,296,061,952.0, sMAPE=75.201%, MASE=759181734083666585649408674493366272.000, AIC=-284.0, BIC=-228.1
    ⛔ Early stopping this config after fold 1 based on AIC (fold AIC=-284.0, best first-fold AIC=-576.4).

=== Config {'order_d': 0, 'order_p': 2, 'order_q': 2, 'seasonal_D': 0, 'seasonal_P': 0, 'seasonal_Q': 1, 'seasonal_period': 12} ===
  Fold 1: Train 2007-04–2017-03, Val 2017-04–2018-03
    Fold 1 MAE(£)=3,644,232,396,458,268,872,588,637,765,632.0, RMSE(£)=215,818,520,610,7

[Parallel(n_jobs=8)]: Done 144 out of 144 | elapsed: 365.0min finished


## Results

In [9]:
# 3) Collect results
results = [r for r in parallel_results if r is not None]

results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values("RMSE_mean").reset_index(drop=True)
    print("\n=== TOP SARIMAX CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===")
    print(results_df.head(10))
    results_df.to_csv("../../results/sarimax_panel_rollingcv_results.csv", index=False)
    print("\nSaved tuning results to ../../results/sarimax_panel_rollingcv_results.csv")
else:
    print("\nNo successful configs to report.")


=== TOP SARIMAX CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===
      model_type      order seasonal_order  folds_used       MAE_mean  \
0  SARIMAX_panel  (0, 0, 0)  (0, 0, 0, 12)           5   11241.085292   
1  SARIMAX_panel  (0, 0, 1)  (0, 0, 0, 12)           5   11063.862389   
2  SARIMAX_panel  (1, 0, 2)  (0, 0, 0, 12)           5   10506.185746   
3  SARIMAX_panel  (0, 0, 2)  (1, 0, 0, 12)           5   10598.713840   
4  SARIMAX_panel  (2, 0, 2)  (0, 0, 0, 12)           5   10648.884055   
5  SARIMAX_panel  (0, 0, 2)  (0, 0, 0, 12)           5   10557.555884   
6  SARIMAX_panel  (1, 0, 0)  (0, 0, 0, 12)           5   11170.541733   
7  SARIMAX_panel  (1, 0, 1)  (0, 0, 0, 12)           5   11274.043400   
8  SARIMAX_panel  (2, 0, 0)  (0, 0, 0, 12)           4   11873.838118   
9  SARIMAX_panel  (0, 1, 0)  (0, 0, 0, 12)           3  109772.093515   

       RMSE_mean  sMAPE_mean  MASE_mean    AIC_mean    BIC_mean  
0   21037.183861    3.893702   0.116321 -642.058319 -588.855415  
1   2